# 03 · Geração fundamentada (RAG): JSON válido, citação e recusa

O modelo recebe os 3 trechos mais parecidos com a pergunta e precisa devolver **somente JSON** no contrato abaixo,
citando os ids dos trechos usados. Se o contexto não responder, deve **recusar**.

```json
{"resposta": "...", "categoria": "buraco_via|calcada|...", "risco": "baixo|medio|alto|nao_se_aplica",
 "fontes": ["T001"], "recusou": false}
```
A validação é feita com Pydantic (`src/zela/esquema.py`): categoria e risco fechados, ids no formato `T000`.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath("../src"))
from zela.base import carregar_trechos
from zela.busca import Buscador
from zela.config import Config
from zela.llm import GeradorHF
from zela.pipeline import PipelineZela

cfg = Config()
print(cfg)
buscador = Buscador(carregar_trechos(), cfg.backend_busca, cfg.modelo_emb)
gerador = GeradorHF(cfg.modelo_llm, None, cfg.max_novos_tokens)
A = PipelineZela("A", gerador)
B = PipelineZela("B", gerador, buscador, cfg.top_k, cfg.limiar)

Config(modelo_llm='Qwen/Qwen2.5-1.5B-Instruct', modelo_emb='intfloat/multilingual-e5-small', backend_busca='e5', adaptador_lora=WindowsPath('C:/Users/guili/Desktop/zelacidade-genai/modelos/lora-zela'), top_k=3, max_novos_tokens=350, semente=42, limiar=0.8511)


W1112 16:09:35.062000 4844 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

C:\Users\guili\Desktop\zelacidade-genai\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\guili\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

... (5 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [2]:
exemplos = [
    "Tem um buraco enorme na minha rua que já derrubou um motoqueiro, o que eu faço?",   # respondível
    "O bueiro da esquina entope toda vez que chove, quem limpa?",                        # sinônimo (bueiro ≈ boca de lobo)
    "Qual é o valor da multa por calçada irregular?",                                   # não consta (mesmo tema!)
    "Caiu um fio da rede elétrica na calçada depois da chuva",                           # emergência
]
for q in exemplos:
    for pipe in (A, B):
        e = pipe.responder(q)
        print(f"\n[{pipe.config}] {q}")
        print("  contexto:", e.contexto, "| recusa por limiar:", e.recusa_por_limiar)
        print("  JSON válido:", e.saida is not None, "|", e.erro_json or "")
        print("  ", json.dumps(e.saida.model_dump() if e.saida else e.texto_bruto, ensure_ascii=False, indent=1)[:700])


[A] Tem um buraco enorme na minha rua que já derrubou um motoqueiro, o que eu faço?
  contexto: [] | recusa por limiar: False
  JSON válido: True | 
   {
 "resposta": "Peça para a prefeitura instalar uma barreira de proteção ou um sinal de alerta próximo ao buraco. Caso seja um problema mais sério, procure ajuda profissional ou entre em contato com a polícia. Não deixe o local ficar desprotegido.",
 "categoria": "obstaculo_sinalizacao",
 "risco": "alto",
 "fontes": [],
 "recusou": false
}



[B] Tem um buraco enorme na minha rua que já derrubou um motoqueiro, o que eu faço?
  contexto: [{'id': 'T001', 'score': 0.8641}, {'id': 'T004', 'score': 0.8623}, {'id': 'T006', 'score': 0.8609}] | recusa por limiar: False
  JSON válido: False | sem_json
   "Recusou: true\n\nFontes: [T001]\n\nCategoria: buraco_via\n\nRisco: alto\n\nOcorrencia: Através do site oficial da Prefeitura de São Paulo, você pode fazer uma denúncia através do número 180. Informe detalhes do problema, incluindo localização exata, descrição do buraco e quaisquer outros fatos relevantes. Além disso, você pode entrar em contato com a Defesa Civil (199) ou o Corpo de Bombeiros (193), dependendo das circunstâncias específicas."



[A] O bueiro da esquina entope toda vez que chove, quem limpa?
  contexto: [] | recusa por limiar: False
  JSON válido: True | 
   {
 "resposta": "O bueiro da esquina entupido pode ser limpo por uma empresa privada contratada pela prefeitura, mas geralmente é responsabilidade do morador verificar e limpar manualmente. Caso seja muito entupido, recomendo entrar em contato com a Secretaria Municipal de Serviços Públicos para obter mais informações.",
 "categoria": "buraco_via",
 "risco": "baixo",
 "fontes": [
  "T002",
  "T003"
 ],
 "recusou": false
}



[B] O bueiro da esquina entope toda vez que chove, quem limpa?
  contexto: [{'id': 'T038', 'score': 0.8577}, {'id': 'T005', 'score': 0.8478}, {'id': 'T006', 'score': 0.8457}] | recusa por limiar: False
  JSON válido: False | sem_json
   "Recusou\n\nFontes: \n- [T005]\n\nO bueiro entupido na esquina deve ser reparado pela Supervisão Técnica de Manutenção. Ela consegue resolver o problema de forma mais eficiente e segura, evitando danos adicionais durante a limpeza."

[A] Qual é o valor da multa por calçada irregular?
  contexto: [] | recusa por limiar: False
  JSON válido: False | sem_json
   "Recusou"



[B] Qual é o valor da multa por calçada irregular?
  contexto: [{'id': 'T021', 'score': 0.8649}, {'id': 'T020', 'score': 0.8558}, {'id': 'T022', 'score': 0.8535}] | recusa por limiar: False
  JSON válido: False | sem_json
   "recusou:true\nfontes:[\"T021\",\"T020\"]\ncategoria:\"obstaculo_sinalizacao\"\nrisco:\"alto\"\n\nA Lei Municipal 15.442/2011 estabelece multas para infrações como a falta de faixa livre em calçadas, mas especificamente quanto à multa por calçada irregular não está explicitamente mencionada. É recomendável consultar diretamente os canais oficiais da Prefeitura de São Paulo para obter informações mais precisas sobre essa questão."



[A] Caiu um fio da rede elétrica na calçada depois da chuva
  contexto: [] | recusa por limiar: False
  JSON válido: False | esquema: ('categoria',) Input should be 'buraco_via', 'calcada', 'iluminacao', 'arvore', 'drenagem', 'emergencia', 'obstaculo_sinalizacao', 'canais', 'plataforma' or 'fora_do_escopo'
   "{\n    \"resposta\": \"Peço desculpas pelo problema. Por favor, entre em contato com a equipe de manutenção do serviço público para verificar e reparar o fio cairido. Caso seja uma emergência, por favor, ligue para o 193.\",\n    \"categoria\": \"burrovia\",\n    \"risco\": \"alto\",\n    \"fontes\": [\"T002\", \"T003\"],\n    \"recusou\": false\n}"



[B] Caiu um fio da rede elétrica na calçada depois da chuva
  contexto: [{'id': 'T030', 'score': 0.8608}, {'id': 'T041', 'score': 0.8542}, {'id': 'T019', 'score': 0.85}] | recusa por limiar: False
  JSON válido: False | sem_json
   "Recusou: true\n\nFontes: [T030]\n\nOrientação:\n\n1. Apareça em local seguro, distante da rede elétrica.\n2. Não toque nos fios ou nas árvores.\n3. Informe a situação à empresa de distribuição de energia elétrica mais próxima.\n4. Mantenha a calçada intacta e siga as recomendações de manutenção após a queda de fios."


## O que observar
- **A (sem contexto)** tende a inventar telefones, prazos e valores com confiança — é a alucinação que queremos medir.
- **B** cita trechos e recusa a pergunta da multa quando os trechos recuperados não têm o valor.
- Falhas de formato (texto fora do JSON, categoria inventada) aparecem em `erro_json` e entram no placar.